# Research: Planta Física water allocation

## Team identification and role rotation

- **Course:** MA2015.202
- **Members:** `____`, `____`, `____`, `____`


We use the supplied baseline genetic algorithm. No operator has been modified for this milestone.

## A1 — Formulation and assumptions

We allocate treated water to four buildings using aggregate March 2025 demands. The building order is fixed:

| Building | Demand Di (m³) | Integer gene ki | Fraction xi = ki / 10 |
|---|---:|---:|---:|
| AULAS 1 | 847.00 | k1 in {0,...,8} | k1 / 10 |
| BIBLIOTECA | 462.00 | k2 in {0,...,8} | k2 / 10 |
| CEDETEC | 284.21 | k3 in {0,...,6} | k3 / 10 |
| AULAS 2 | 683.00 | k4 in {0,...,8} | k4 / 10 |

The total observed March 2025 demand is:

$$ \sum_{i=1}^{4} D_i = 847.00 + 462.00 + 284.21 + 683.00 = 2276.21 \ \text{m}^3. $$

For a candidate chromosome $k=(k_1,k_2,k_3,k_4)$, we define:

$$ T = \sum_{i=1}^{4} D_i \, k_i / 10 $$

$$ P = 2276.21 - T $$

$$ E = \max(0, \, T - 1410) $$

where:

- $T$ = treated water allocated, in m³
- $P$ = potable water still required, in m³
- $E$ = excess treated water above capacity, in m³
- $C = 1410 \ \text{m}^3$ is the teaching capacity

The objective is to minimize potable water $P$, subject to:

$$ 0 \le k_1, k_2, k_4 \le 8, \qquad 0 \le k_3 \le 6, $$

and

$$ T \le C = 1410 \ \text{m}^3. $$

We use a numerical tolerance of $10^{-8} \ \text{m}^3$. The evaluator rejects invalid gene domains without silent rounding or repair.

We rank candidates with fitness $F$, where **lower is better**:

$$ F = \begin{cases} \dfrac{P}{2276.21}, & \text{if } E \le 10^{-8} \\[6pt] 1 + \dfrac{E}{1410}, & \text{if } E > 10^{-8} \end{cases} $$

A feasible candidate has $F \le 1$. An infeasible candidate has $F > 1$.

### Observed data versus teaching assumptions

**Observed data:** the four March 2025 demands come from the supplied records. The metadata reports no imputation, zero, or possible-anomaly flags.

**Teaching assumptions:** the capacity $C = 1410 \ \text{m}^3$ and the per-building integer gene limits are teaching assumptions from the instructor calibration. They are not validated hydraulic or sanitary limits. Results describe an academic monthly model, not an operational recommendation.

## A2 — Hand calculation and verification

### Candidate 1: uniform 60% reuse, k = (6, 6, 6, 6)

We calculate by hand:

$$ T = \frac{6}{10}(847 + 462 + 284.21 + 683) = 0.6 \times 2276.21 = 1365.726 \ \text{m}^3. $$

$$ P = 2276.21 - 1365.726 = 910.484 \ \text{m}^3. $$

$$ E = \max(0, \, 1365.726 - 1410) = 0 \ \text{m}^3. $$

Since $E = 0$, the candidate is feasible. The capacity margin is:

$$ 1410 - 1365.726 = 44.274 \ \text{m}^3. $$

The fitness is:

$$ F = \frac{910.484}{2276.21} = 0.4. $$

So $k = (6,6,6,6)$ gives:

- $T = 1365.726 \ \text{m}^3$
- $P = 910.484 \ \text{m}^3$
- $E = 0 \ \text{m}^3$
- Feasible: yes
- $F = 0.4$

### Candidate 2: maximum gene levels, k = (8, 8, 6, 8)

This candidate is valid at the gene level because all genes are inside their domains. However, it must also satisfy capacity.

$$ T = 847(0.8) + 462(0.8) + 284.21(0.6) + 683(0.8) $$

$$ T = 677.6 + 369.6 + 170.526 + 546.4 = 1764.126 \ \text{m}^3. $$

$$ P = 2276.21 - 1764.126 = 512.084 \ \text{m}^3. $$

$$ E = \max(0, \, 1764.126 - 1410) = 354.126 \ \text{m}^3. $$

Since $E > 0$, the candidate is infeasible. Its fitness is penalized:

$$ F = 1 + \frac{354.126}{1410} = 1.2511531914893617. $$

The maximum plan uses less potable water, $P = 512.084 \ \text{m}^3$, compared with $P = 910.484 \ \text{m}^3$ for $k=(6,6,6,6)$. However, it requires $T = 1764.126 \ \text{m}^3$, which exceeds the teaching capacity by $E = 354.126 \ \text{m}^3$. Therefore, the capacity constraint $T \le 1410$ is violated. The evaluator marks the candidate infeasible and assigns $F > 1$. In this model, a feasible solution within capacity is preferred over an infeasible solution that would otherwise reduce potable water more.

In [1]:
import numpy as np

# Manual candidates
manual = np.array([6, 6, 6, 6])
maximum = LIMITS.copy()  # [8, 8, 6, 8]

print("Manual calculation check")
print("-" * 60)
for label, genes in [("(6,6,6,6)", manual), ("(8,8,6,8)", maximum)]:
    m = evaluate(genes)
    print(f"{label}: genes={genes}, T={m['treated']:.6f} m3, "
          f"P={m['potable']:.6f} m3, E={m['excess']:.6f} m3, "
          f"feasible={m['feasible']}, F={m['fitness']:.12f}")

# Assertions for (6,6,6,6)
m1 = evaluate(manual)
assert np.isclose(m1['treated'], 1365.726, atol=1e-8)
assert np.isclose(m1['potable'], 910.484, atol=1e-8)
assert np.isclose(m1['excess'], 0.0, atol=1e-8)
assert m1['feasible']
assert np.isclose(m1['fitness'], 0.4, atol=1e-12)

# Assertions for (8,8,6,8)
m2 = evaluate(maximum)
assert np.isclose(m2['treated'], 1764.126, atol=1e-8)
assert np.isclose(m2['potable'], 512.084, atol=1e-8)
assert np.isclose(m2['excess'], 354.126, atol=1e-8)
assert not m2['feasible']
assert np.isclose(m2['fitness'], 1 + 354.126/1410, atol=1e-12)

print("\nAll manual calculations match the evaluator within tolerance.")

NameError: name 'LIMITS' is not defined